# పాఠం 09 - మెటాకాగ్నిషన్ డిజైన్ నమూనా


## సెటప్

ఈ నోట్‌బుక్ Microsoft Agent Framework ఉపయోగించి మెటాకాగ్నిషన్ డిజైన్ ప్యాటర్న్‌ని ప్రదర్శిస్తుంది.

**ముందస్తు అవసరాలు:**
- Azure OpenAI deployment పర్యావరణ వేరియబుల్స్ ద్వారా కాన్ఫిగర్ చేయబడినది
- Azure CLI లో లాగిన్ అయి ఉండాలి (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## మెటాకాగ్నిషన్ అంటే ఏమిటి?

మెటాకాగ్నిషన్ అంటే **ఆలోచనపై ఆలోచించడం**. AI ఏజెంట్ల సందర్భంలో, దీని అర్థం అటువంటి ఏజెంట్లను నిర్మించడం:

- **స్వయంగా ఆత్మపరిశీలన చేయడం** వారి స్వంత ఫలితాలు మరియు తర్క ప్రక్రియపై
- **దోషాలను గుర్తించడం** మరియు నిశ్శబ్దంగా విఫలమవకుండా సమర్థంగా పునరుద్ధరించుకోవడం
- **మూల్యాంకనం చేయడం** వారి సమాధానాలు సంపూర్ణమా మరియు సహాయకమా అని
- **అనుకూలీకరించడం** వారి వ్యూహాన్ని ప్రారంభిక విధానం పనిచేయకపోతే మార్చుకోవడం (ఉదాహరణకు, బ్యాకప్ సిస్టమ్‌కు తిరిగి వెళ్ళడం)

ఒక మెటాకాగ్నిటివ్ ఏజెంట్ కేవలం ప్రశ్నలకు మాత్రమే సమాధానం ఇవ్వదు — అది తన స్వంత పనితీరును పర్యవేక్షించి వెంటనే సర్దుబాటు చేస్తుంది.


## ప్రధాన మరియు బ్యాకప్ సాధనాలు

మెటాకాగ్నిషన్‌లో ఒక సాధారణ నమూనా **ప్రత్యామ్నాయ వ్యూహం**.

 ఏజెంట్ మొదట ఒక ప్రధాన సాధనాన్ని ప్రయత్నిస్తాడు; అది విఫలమైతే (ఉదా., 404 తప్పు), ఏజెంట్ వైఫల్యాన్ని గుర్తించి పారదర్శకంగా బ్యాకప్ సాధనానికి మారుతుంది.

ఇది వాస్తవ ప్రపంచ వ్యవస్థలను ప్రతిబింబిస్తుంది, అక్కడ ప్రధాన సేవలు అందుబాటులో లేకపోవచ్చు మరియు ఏజెంట్లు ప్రత్యామ్నాయ మార్గాన్ని ఎంచుకోవడానికి ముందు సమస్యను స్వయంగా నిర్ధారించుకోవాలి.

కింద మేము రెండు విమానాల శోధన సాధనాలను నిర్వచిస్తాము:
- **ప్రాథమిక** — Paris, Tokyo, మరియు Barcelona ను కవర్ చేస్తుంది
- **బ్యాకప్** — Berlin, Sydney, మరియు New York City ను కవర్ చేస్తుంది


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## లోపాల పునరుద్ధరణతో స్వ-పరిశీలన ఏజెంట్

క్రింద ఉన్న ఏజెంట్‌ను మొదట ప్రాధమిక ఫ్లైట్ సిస్టమ్‌ను ప్రయత్నించాలని, వైఫల్యాలను గుర్తించి పారదర్శకంగా బ్యాకప్ సిస్టమ్‌కు తిరిగి మారాలని సూచించారు. ప్రతి స్పందన తర్వాత ఇది సంక్షిప్తంగా స్వీయ-మూల్యాంకనం చేసి వాడుకరి ప్రశ్నకు పూర్తిగా సమాధానం ఇచ్చిందో లేదో చూసుకుంటుంది.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## స్వ-మూల్యాంకన నమూనా

మెటాకాగ్నిషన్ యొక్క మరో పార్శ్వం **స్వ-మూల్యాంకనం**: వేరే ఏజెంట్ (లేదా రెండవ పాస్‌లో అదే ఏజెంట్) ఒక ప్రతిస్పందనను సమగ్రత, ఖచ్చితత్వం, మరియు ఉపయోగకరత కోసం సమీక్షిస్తుంది.

క్రింద మేము ఒక `ResponseEvaluator` ఏజెంట్‌ను సృష్టిస్తాము, అది ప్రయాణ ఏజెంట్ ప్రతిస్పందనలను మూడు పరిమాణాలలో స్కోర్ చేస్తుంది.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ఉపయోగించి **మెటాకాగ్నిటివ్ ఏజెంట్లు** ఎలా నిర్మించాలో నేర్చుకున్నారు:

- **స్వ-పరిశీలన**: తమ స్వంత తార్కిక ప్రక్రియలను పర్యవేక్షించి, ఏమి జరిగిందో పారదర్శకంగా తెలియచేసే ఏజెంట్లు.
- **ఫాల్బ్యాక్‌లతో లోప పునరుద్ధరణ**: ప్రాధమిక + బ్యాకప్ టూల్ నమూనా, ఇందులో ఏజెంట్ వైఫల్యాలను (ఉదా., 404 errors) గుర్తించి స్వయంచాలకంగా ప్రత్యామ్నాయ మూలాన్ని ప్రయత్నిస్తుంది.
- **స్వ-మూల్యాంకనం**: సంపూర్ణత, ఖచ్చితత్వం మరియు సహాయకత్వం కోసం ప్రతిస్పందనలకు స్కోరు ఇచ్చే ప్రత్యేక మూల్యాంకన ఎజెంట్.

ఈ నమూనాలు ఏజెంట్లను మరింత బలవంతంగా, పారదర్శకంగా మరియు నమ్మకంగా మార్చుతాయి — ప్రొడక్షన్ అమలులో కీలకమైన లక్షణాలు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
నిరాకరణ:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా లోపాలు ఉండవచ్చు అని దయచేసి గమనించండి. మూల పత్రాన్ని దాని స్వదేశీ భాషలోని ప్రతిని అధికారిక మూలంగా పరిగణించవలెను. ముఖ్యమైన సమాచారానికి వృత్తిపరమైన మానవ అనువాదం సూచించబడుతుంది. ఈ అనువాదాన్ని ఉపయోగించడంతో ఏర్పడిన ఏవైనా అవగాహన లోపాలు లేదా తప్పుగా అర్థం చేసుకోవడాలకు మేము బాధ్యులు కాదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
